# RSNA Knee MRI — Huấn luyện trên Colab (A100 80GB)

Text-guided knowledge distillation: **huấn luyện** bằng ảnh + report,
**inference** chỉ cần ảnh + metadata series.

Notebook này chỉ *gọi* các hàm trong package `knee_mri`; toàn bộ logic nằm trong
`src/` nên chạy được cả ngoài notebook và có test bao phủ.

## Cách dùng

1. Đặt `MODE = 'pilot'` (cell 2) → chạy **toàn bộ** notebook từ trên xuống. Pilot dùng
   *toàn bộ 58 study gold* + một mẫu nhỏ study không nhãn, artifacts ghi riêng vào
   `artifacts/pilot/`. Cuối notebook có ước lượng thời gian cho bản đầy đủ.
2. Nếu pilot cho AUC > ngẫu nhiên và thời gian chấp nhận được → đổi `MODE = 'full'`,
   **Restart runtime**, chạy lại từ đầu.
3. Runtime bị ngắt giữa chừng? Chạy lại từ đầu: mọi bước precompute bỏ qua file đã
   có, còn huấn luyện tự tiếp tục từ `last.pt` (`resume=True`).

Drive cần có: `dataset/` (dữ liệu RSNA). `artifacts/` và `vendor/` sẽ tự được tạo.

In [3]:
# 0. Bootstrap: lấy code, cài package, nạp cấu hình — chạy được cả Colab lẫn local.
#    Code luôn được clone/pull vào /content (Colab) rồi `pip install -e`, nên không còn
#    phụ thuộc vào việc bạn đã tự chép src/ lên Drive. Chỉ dataset/, artifacts/ và
#    vendor/ nằm trên Drive (bền vững giữa các lần runtime bị ngắt).
import logging, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hoangtung386/Att-kneenet.git'
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/Challenge_RNSA_KNEE')
    CODE = Path('/content/Att-kneenet')
    if (CODE / '.git').is_dir():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[viz,eval]'], check=True)
else:
    CODE = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
    DRIVE_ROOT = CODE
sys.path.insert(0, str(CODE / 'src'))

assert (DRIVE_ROOT / 'dataset' / 'train.csv').is_file(), (
    f'Không thấy dataset tại {DRIVE_ROOT / "dataset"} — hãy đặt dataset RSNA vào đó.'
)
print('code   :', CODE)
print('dataset:', DRIVE_ROOT / 'dataset')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
code   : /content/Att-kneenet
dataset: /content/drive/MyDrive/Challenge_RNSA_KNEE/dataset


In [4]:
# 1. Chế độ chạy — ĐỔI Ở ĐÂY
MODE = 'pilot'        # 'pilot' | 'full'
PILOT_EXTRA = 150     # số study không nhãn thêm vào pilot (toàn bộ gold luôn được giữ)
PILOT_EPOCHS = 3
assert MODE in ('pilot', 'full')

In [5]:
# 2. Đăng nhập HuggingFace bằng Colab Secret 'HF_TOKEN'
#    Cần đã bấm "Agree" cho google/gemma-3-12b-it (model gated) bằng đúng tài khoản này.
import os
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Đã đăng nhập HuggingFace.')

Đã đăng nhập HuggingFace.


## Cấu hình

Cấu hình là dataclass bất biến nạp từ YAML. Ghi đè ở đây **có tác dụng thật** vì
`cfg` được truyền tường minh xuống mọi hàm — khác với cách dùng hằng số cấp
module, nơi `from module import CONSTANT` sao chép giá trị lúc import và mọi ghi
đè sau đó bị nuốt im lặng.

Pilot ghi artifacts vào `artifacts/pilot/` (tách hẳn checkpoint, mask, cache) để
`resume=True` của bản full không vô tình nối tiếp checkpoint của pilot.

In [6]:
# 3. Nạp cấu hình
import os
from knee_mri.config import load_config
from knee_mri.utils.logging import setup_logging
from knee_mri.utils.seed import set_seed
import torch

BASE_ARTIFACTS = DRIVE_ROOT / 'artifacts'
ARTIFACTS = BASE_ARTIFACTS / ('pilot' if MODE == 'pilot' else '')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Weights 3DINO (~1.4GB) luôn nằm ở artifacts gốc và dùng chung cho pilot lẫn full. Đặt
# dino.weight_file thành đường dẫn TUYỆT ĐỐI: pathlib bỏ qua artifacts_root khi gặp đường dẫn
# tuyệt đối, nên mọi nơi gọi cfg.dino.weights_path(...) đều trỏ về file này.
# (Không dùng symlink: Google Drive không hỗ trợ — OSError 95.)
DINO_WEIGHTS = BASE_ARTIFACTS / load_config('colab').dino.weight_file

overrides = {
    'paths': {
        'data_root': str(DRIVE_ROOT / 'dataset'),
        'artifacts_root': str(ARTIFACTS),
        'dino_code_dir': str(DRIVE_ROOT / 'vendor' / '3DINO'),
    },
    'dino': {'weight_file': str(DINO_WEIGHTS)},
}
if MODE == 'pilot':
    overrides['train'] = {'epochs': PILOT_EPOCHS}

cfg = load_config('colab', overrides=overrides)
cfg.paths.ensure()
setup_logging(logging.INFO, log_dir=cfg.paths.logs)
set_seed(cfg.seed)

print('MODE         :', MODE)
print('dataset      :', cfg.paths.data_root)
print('artifacts    :', cfg.paths.artifacts_root)
print('target_shape :', cfg.data.target_shape)
print('VLM          :', cfg.vlm.model_id, '| epochs:', cfg.train.epochs)
print('thiết bị     :', 'cuda' if torch.cuda.is_available() else 'cpu')

MODE         : pilot
dataset      : /content/drive/MyDrive/Challenge_RNSA_KNEE/dataset
artifacts    : /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot
target_shape : (64, 128, 128)
VLM          : google/gemma-3-12b-it | epochs: 3
thiết bị     : cuda


In [7]:
# 4. Clone 3DINO (chứa package `dinov2`) và tải weights — URL, tên file, thư mục lấy từ cfg.
#    License CC BY-NC-ND: clone nguyên gốc, KHÔNG sửa code trong vendor/.
#    requirements của 3DINO được cài với *constraints* khóa torch/transformers/numpy ở
#    đúng bản Colab đang có, để chúng không bị hạ/nâng cấp âm thầm.
import glob, shutil
from huggingface_hub import snapshot_download

REPO = cfg.paths.dino_code_dir
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', cfg.dino.repo_url, str(REPO)], check=True)

PROTECTED = {'torch', 'torchvision', 'torchaudio', 'transformers', 'tokenizers', 'numpy',
             'huggingface-hub', 'accelerate', 'safetensors'}
def frozen_versions():
    out = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
    return dict(l.split('==', 1) for l in out.splitlines() if '==' in l)
before = {k: v for k, v in frozen_versions().items() if k.lower() in PROTECTED}
constraints = Path('/content/constraints.txt')
constraints.write_text('\n'.join(f'{k}=={v}' for k, v in before.items()))

pip = [sys.executable, '-m', 'pip', 'install', '-q']
result = subprocess.run(pip + ['-c', str(constraints), '-r', str(REPO / 'requirements.txt')])
if result.returncode != 0:
    print('! requirements 3DINO xung đột với bản torch/transformers của Colab -> cài --no-deps.')
    print('  Nếu import dinov2 bên dưới báo thiếu module, cài thủ công module đó.')
    subprocess.run(pip + ['--no-deps', '-r', str(REPO / 'requirements.txt')])

# Tải weights một lần vào Drive (đường dẫn lấy từ cfg, xem cell 3).
dino_weights = cfg.dino.weights_path(cfg.paths.artifacts_root)
if not dino_weights.exists():
    downloaded = snapshot_download('AICONSlab/3DINO-ViT', token=HF_TOKEN)
    shutil.copy(glob.glob(os.path.join(downloaded, cfg.dino.weight_file.rsplit('/', 1)[-1]))[0], dino_weights)

# Kiểm tra torch/transformers không bị đổi và dinov2 import được
after = frozen_versions()
changed = {k: (v, after.get(k)) for k, v in before.items() if after.get(k) != v}
assert not changed, f'Dependency bị đổi bởi cài 3DINO: {changed} — Restart runtime và kiểm tra lại.'
sys.path.insert(0, str(REPO))
from dinov2.configs import load_and_merge_config_3d  # noqa: F401  (smoke test import)
print('3DINO code   :', REPO)
print('3DINO weights:', dino_weights)

! requirements 3DINO xung đột với bản torch/transformers của Colab -> cài --no-deps.
  Nếu import dinov2 bên dưới báo thiếu module, cài thủ công module đó.
3DINO code   : /content/drive/MyDrive/Challenge_RNSA_KNEE/vendor/3DINO
3DINO weights: /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/3dino_vit_weights.pth


In [8]:
# 5. Nạp catalog (index CSV MỘT LẦN, tra cứu O(1) sau đó) và chọn tập study
from knee_mri.data.catalog import StudyCatalog
from knee_mri.data.splits import pilot_subset

catalog = StudyCatalog.from_csv(cfg.paths.train_csv, cfg.paths.train_series_csv)
all_studies = catalog.studies_with_series()
n_gold = len(catalog.studies_with_gold())

# Pilot = TOÀN BỘ gold (làm validation) + mẫu ngẫu nhiên cố định seed. Không cắt 50 study
# đầu như trước — cách đó chỉ giữ lại 1 study gold.
subset = pilot_subset(catalog, extra=PILOT_EXTRA, seed=cfg.seed) if MODE == 'pilot' else all_studies

print(catalog)
print(f'MODE={MODE}: {len(subset)} study xử lý ({n_gold} gold làm validation)')
assert n_gold >= 10, 'Quá ít study gold có series — không thể validation.'

INFO:knee_mri.data.catalog:Catalog: 4407 study, 24371 series, 4407 report, 58 study có nhãn gold.


03:42:38 | INFO    | knee_mri.data.catalog        | Catalog: 4407 study, 24371 series, 4407 report, 58 study có nhãn gold.
StudyCatalog(studies=4407, reports=4407, gold=58)
MODE=pilot: 208 study xử lý (58 gold làm validation)


## Kiểm tra trước khi chạy (preflight)

Chặn lỗi rẻ-mà-đắt: token HF chưa được cấp quyền, `guidance_dim` lệch với VLM thật,
thiếu GPU, hết dung lượng Drive. Tất cả đều dừng *trước* khi tốn giờ GPU.

In [9]:
# 6. Preflight — dừng ở đây nếu có vấn đề, trước khi tốn giờ GPU
from knee_mri.pipeline import run_preflight

preflight = run_preflight(cfg, catalog, subset, hf_token=HF_TOKEN)
print(preflight.info)
preflight.raise_if_failed()
print('Preflight OK.')

INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-12b-it/resolve/main/config.json "HTTP/1.1 200 OK"


03:42:38 | INFO    | httpx                        | HTTP Request: HEAD https://huggingface.co/google/gemma-3-12b-it/resolve/main/config.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/sam-vit-huge/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


03:42:39 | INFO    | httpx                        | HTTP Request: HEAD https://huggingface.co/facebook/sam-vit-huge/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/sam-vit-huge/87aecf0df4ce6b30cd7de76e87673c49644bdf67/config.json?%2Ffacebook%2Fsam-vit-huge%2Fresolve%2Fmain%2Fconfig.json=&etag=%2297290178b40b269ecc23cb13c50e44c41c222df0%22 "HTTP/1.1 200 OK"


03:42:39 | INFO    | httpx                        | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/sam-vit-huge/87aecf0df4ce6b30cd7de76e87673c49644bdf67/config.json?%2Ffacebook%2Fsam-vit-huge%2Fresolve%2Fmain%2Fconfig.json=&etag=%2297290178b40b269ecc23cb13c50e44c41c222df0%22 "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


03:42:42 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


INFO:knee_mri.pipeline.preflight:Preflight: OK | {'gpu': 'NVIDIA A100-SXM4-80GB, 85 GB', 'vlm_hidden_size': 3840, 'mask_gb': 0.872415232, 'free_gb': 238.878982144, 'needed_gb': 6.1341398016}


03:42:42 | INFO    | knee_mri.pipeline.preflight  | Preflight: OK | {'gpu': 'NVIDIA A100-SXM4-80GB, 85 GB', 'vlm_hidden_size': 3840, 'mask_gb': 0.872415232, 'free_gb': 238.878982144, 'needed_gb': 6.1341398016}
{'gpu': 'NVIDIA A100-SXM4-80GB, 85 GB', 'vlm_hidden_size': 3840, 'mask_gb': 0.872415232, 'free_gb': 238.878982144, 'needed_gb': 6.1341398016}
Preflight OK.


In [10]:
# # 7. (Tùy chọn) So sánh các chế độ chuẩn hóa trên mẫu nhỏ — chỉ chạy ở pilot
# RUN_NORM_ANALYSIS = (MODE == 'pilot')
# if RUN_NORM_ANALYSIS:
#     out = subprocess.run(
#         [sys.executable, str(CODE / 'scripts' / 'analyze_normalization.py'),
#          '--env', 'colab', '--max-series', '60'],
#         capture_output=True, text=True, cwd=str(CODE),
#     )
#     print(out.stdout[-3000:] or out.stderr[-3000:])

In [11]:
# 8. Đo thời gian từng bước + tự thử lại khi lỗi thoáng qua (OOM tạm thời, mất kết nối Drive).
#    Mọi bước precompute bỏ qua file đã có, nên thử lại không làm lại phần đã xong.
#    Lỗi cấu hình/thiếu asset (ImportError, thiếu 3DINO...) thì dừng ngay.
from knee_mri.pipeline import StageTimer

timer = StageTimer()

## S1 — Mask ROI và weak label

In [12]:
# 9. Sinh mask ROI bằng SAM
from knee_mri.pipeline import generate_masks
timer.run('masks', generate_masks, cfg, catalog, subset, device='cuda')

INFO:knee_mri.models.sam_masker:Đang nạp SAM facebook/sam-vit-huge...


03:43:14 | INFO    | knee_mri.models.sam_masker   | Đang nạp SAM facebook/sam-vit-huge...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/facebook/sam-vit-huge/revision/main "HTTP/1.1 200 OK"


03:43:14 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/facebook/sam-vit-huge/revision/main "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/facebook/sam-vit-huge/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


03:43:14 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/facebook/sam-vit-huge/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


... (45 outputs left out)

07:42:51 | INFO    | knee_mri.pipeline.masks      | S1 hoàn tất: 832 mask mới tại /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot/masks


INFO:knee_mri.pipeline.runner:[masks] xong sau 240.2 phút


07:42:51 | INFO    | knee_mri.pipeline.runner     | [masks] xong sau 240.2 phút


832

In [13]:
# 10. Sinh weak label từ report
#     use_vlm=True dùng VLM đọc hiểu (lỗi từng study tự lùi về luật từ khóa).
from knee_mri.pipeline import precompute_weak_labels
timer.run('weak_labels', precompute_weak_labels, cfg, catalog, subset, use_vlm=True, device='cuda')

INFO:knee_mri.labeling.llm_based:Đang nạp VLM google/gemma-3-12b-it (dtype=bfloat16)...


07:42:51 | INFO    | knee_mri.labeling.llm_based  | Đang nạp VLM google/gemma-3-12b-it (dtype=bfloat16)...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


07:42:51 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/gemma-3-12b-it/resolve/96b6f1eccf38110c56df3a15bffe176da04bfd80/processor_config.json "HTTP/1.1 200 OK"


07:42:51 | INFO    | httpx                        | HTTP Request: HEAD https://huggingface.co/google/gemma-3-12b-it/resolve/96b6f1eccf38110c56df3a15bffe176da04bfd80/processor_config.json "HTTP/1.1 200 OK"


... (74 outputs left out)

08:24:00 | INFO    | knee_mri.pipeline.weak_labels | Nhãn yếu hoàn tất: 208 file tại /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot/weak_labels


INFO:knee_mri.pipeline.runner:[weak_labels] xong sau 41.1 phút


08:24:00 | INFO    | knee_mri.pipeline.runner     | [weak_labels] xong sau 41.1 phút


208

In [14]:
# 10b. Chấm nhãn yếu vừa sinh (VLM) so với nhãn gold, đặt cạnh luật từ khóa.
#      Nhãn yếu quyết định trần chất lượng của 4349 study không có nhãn thật, nên kiểm tra ở đây
#      trước khi tốn giờ GPU precompute teacher/guidance và huấn luyện.
from knee_mri.labeling import (cached_weak_labels, format_labeler_report, keyword_labeler,
                               score_labeler)

gold_in_subset = [u for u in subset if catalog.gold(u) is not None]
cached = score_labeler(catalog, cached_weak_labels(cfg), gold_in_subset)
baseline = score_labeler(catalog, lambda u: keyword_labeler(catalog.report(u)), gold_in_subset)
print('Nhãn yếu đã cache (VLM, lỗi từng study đã lùi về luật từ khóa):')
print(format_labeler_report(cached))
print(f'\nBaseline luật từ khóa: macro-F1 = {baseline.macro_f1:.3f}')
if cached.macro_f1 < baseline.macro_f1:
    print('! Nhãn VLM KÉM hơn luật từ khóa trên gold — kiểm tra prompt/VLM trước khi train; '
          'hoặc đặt use_vlm=False ở cell 10 và chạy lại.')

Nhãn yếu đã cache (VLM, lỗi từng study đã lùi về luật từ khóa):
nhãn               dương   prec    rec     F1
ACL                   24   0.67   1.00   0.80
MCL                    9   0.38   0.89   0.53
Medial Meniscus       26   0.69   0.96   0.81
Lateral Meniscus      23   0.65   0.87   0.74
Medial OA             15   0.83   1.00   0.91
Lateral OA            11   0.53   0.73   0.62
PF OA                 21   0.82   0.67   0.74
Effusion              35   0.69   1.00   0.81
Synovitis             27   0.65   0.63   0.64
Baker's               12   0.69   0.92   0.79
Contusion             19   0.67   0.84   0.74
Fracture              18   0.68   0.83   0.75
micro                240   0.66   0.87   0.75
macro-F1 = 0.740 trên 58 study gold

Baseline luật từ khóa: macro-F1 = 0.555


## S2 — Precompute teacher rồi huấn luyện student

In [15]:
# 11. Đặc trưng ảnh từ 3DINO-ViT (teacher đóng băng → cache một lần)
from knee_mri.pipeline import precompute_teacher_features
timer.run('teacher_feats', precompute_teacher_features, cfg, catalog, subset, device='cuda')

/content/drive/MyDrive/Challenge_RNSA_KNEE/vendor/3DINO/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/drive/MyDrive/Challenge_RNSA_KNEE/vendor/3DINO/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/drive/MyDrive/Challenge_RNSA_KNEE/vendor/3DINO/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
INFO:knee_mri.models.teachers.dino3d:Đang nạp 3DINO-ViT từ /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/3dino_vit_weights.pth


08:24:14 | INFO    | knee_mri.models.teachers.dino3d | Đang nạp 3DINO-ViT từ /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/3dino_vit_weights.pth


INFO:dinov2:using MLP layer as FFN


08:24:14 | INFO    | dinov2                       | using MLP layer as FFN


INFO:dinov2:Take key teacher in provided checkpoint dict


08:24:54 | INFO    | dinov2                       | Take key teacher in provided checkpoint dict


... (21 outputs left out)

08:31:58 | INFO    | knee_mri.pipeline.features   | Teacher hoàn tất: 832 đặc trưng tại /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot/teacher_feats


INFO:knee_mri.pipeline.runner:[teacher_feats] xong sau 8.0 phút


08:31:58 | INFO    | knee_mri.pipeline.runner     | [teacher_feats] xong sau 8.0 phút


832

In [16]:
# 12. Guidance từ VLM đa modality (ảnh + report → G)
#     KHÔNG bật allow_zero_fallback: guidance toàn 0 làm loss contrastive vô nghĩa.
#     Lỗi thoáng qua được thử lại; lỗi lặp lại thì dừng để bạn xử lý.
from knee_mri.pipeline import precompute_guidance
timer.run('guidance', precompute_guidance, cfg, catalog, subset, device='cuda')

INFO:knee_mri.models.teachers.gemma_guidance:Đang nạp VLM guidance google/gemma-3-12b-it...


08:31:58 | INFO    | knee_mri.models.teachers.gemma_guidance | Đang nạp VLM guidance google/gemma-3-12b-it...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


08:31:58 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


08:31:58 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


... (22 outputs left out)

08:37:19 | INFO    | knee_mri.pipeline.features   | Guidance hoàn tất: 208 vector tại /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot/guidance


INFO:knee_mri.pipeline.runner:[guidance] xong sau 5.4 phút


08:37:19 | INFO    | knee_mri.pipeline.runner     | [guidance] xong sau 5.4 phút


208

In [17]:
# 13. Kiểm tra cache trước khi huấn luyện — thiếu guidance/teacher sẽ bị thay bằng vector 0
#     mà không báo lỗi, nên chặn ở đây.
from knee_mri.data.dataset import describe_cache, missing_cache

print(describe_cache(cfg, subset))
gaps = missing_cache(cfg, subset)
assert not gaps, f'Thiếu cache (số study): {gaps}. Chạy lại các cell 9-12.'

{'studies': 208, 'masks': 208, 'teacher_feats': 208, 'guidance': 208, 'weak_labels': 208}


In [18]:
# 14. Chia tập: study gold = validation cố định, phần còn lại (trong subset) = train
from knee_mri.data.dataset import KneeDataset
from knee_mri.data.splits import make_split
from knee_mri.training.trainer import train

split = make_split(catalog, seed=cfg.seed, subset=subset)
print(split)
assert len(split.val) >= 10, f'Validation chỉ có {len(split.val)} study — AUC không có nghĩa.'

train_ds = KneeDataset(split.train, catalog, cfg)
val_ds = KneeDataset(split.val, catalog, cfg)

INFO:knee_mri.data.splits:Split(train=150, val=58) | gold trong val: 58


08:37:20 | INFO    | knee_mri.data.splits         | Split(train=150, val=58) | gold trong val: 58
Split(train=150, val=58)


In [19]:
# 15. Huấn luyện student. resume=True: nếu runtime bị ngắt, chạy lại cell này (hoặc cả
#     notebook) để tiếp tục từ artifacts/<...>/checkpoints/last.pt trên Drive.
result = train(cfg, train_ds, val_ds, device='cuda', resume=True)
print('Macro AUC tốt nhất:', result.best_metric)
print('Checkpoint        :', result.best_checkpoint)
epoch_seconds = (
    sum(r['seconds'] for r in result.history) / len(result.history) if result.history else None
)

INFO:knee_mri.training.trainer:Bắt đầu huấn luyện trên cuda | 150 study train, 58 study val | 3 epoch


08:37:20 | INFO    | knee_mri.training.trainer    | Bắt đầu huấn luyện trên cuda | 150 study train, 58 study val | 3 epoch


INFO:knee_mri.training.trainer:epoch 0 step 0/19 | total=5.2331 kd=0.9705 contrast=2.0986 cls=1.6729 auc=0.9821 | lr=5.00e-05


08:37:39 | INFO    | knee_mri.training.trainer    | epoch 0 step 0/19 | total=5.2331 kd=0.9705 contrast=2.0986 cls=1.6729 auc=0.9821 | lr=5.00e-05


INFO:knee_mri.training.trainer:epoch 0 xong sau 94.9s | total=4.6230 kd=0.7081 contrast=2.0634 cls=1.3533 auc=0.9966


08:38:55 | INFO    | knee_mri.training.trainer    | epoch 0 xong sau 94.9s | total=4.6230 kd=0.7081 contrast=2.0634 cls=1.3533 auc=0.9966


... (12 outputs left out)

INFO:knee_mri.training.trainer:epoch 2 xong sau 108.9s | total=4.1787 kd=0.5405 contrast=1.8844 cls=1.2581 auc=0.9915


08:43:33 | INFO    | knee_mri.training.trainer    | epoch 2 xong sau 108.9s | total=4.1787 kd=0.5405 contrast=1.8844 cls=1.2581 auc=0.9915


INFO:knee_mri.training.trainer:Validation:
macro AUC = 0.5366 (trên 12 nhãn)
    Effusion           0.7230
    MCL                0.6644
    ACL                0.6219
    Medial OA          0.5969
    Baker's            0.5571
    Lateral Meniscus   0.5286
    Synovitis          0.5143
    Fracture           0.4903
    PF OA              0.4659
    Lateral OA         0.4603
    Contusion          0.4332
    Medial Meniscus    0.3834


08:44:10 | INFO    | knee_mri.training.trainer    | Validation:
macro AUC = 0.5366 (trên 12 nhãn)
    Effusion           0.7230
    MCL                0.6644
    ACL                0.6219
    Medial OA          0.5969
    Baker's            0.5571
    Lateral Meniscus   0.5286
    Synovitis          0.5143
    Fracture           0.4903
    PF OA              0.4659
    Lateral OA         0.4603
    Contusion          0.4332
    Medial Meniscus    0.3834
Macro AUC tốt nhất: 0.5387751011566557
Checkpoint        : /content/drive/MyDrive/Challenge_RNSA_KNEE/artifacts/pilot/checkpoints/student_best.pt


## Đánh giá độc lập

AUC bên dưới tính lại từ checkpoint tốt nhất bằng đường **inference thật**
(`Predictor`, chỉ ảnh + metadata) trên study gold — không dùng cache teacher/guidance,
nên là con số trung thực nhất về chất lượng lúc nộp bài.

In [20]:
# 16. Đánh giá checkpoint tốt nhất trên các study gold
from knee_mri.inference.predictor import Predictor, evaluate_gold
from knee_mri.training.metrics import format_auc_report

predictor = Predictor.from_checkpoint(cfg, checkpoint=result.best_checkpoint, device='cuda')
evaluation = evaluate_gold(predictor, catalog, split.val, log_every=0)
print(f'{len(evaluation.study_uids)} study gold')
print(format_auc_report(evaluation.report))
if evaluation.report.macro < 0.55:
    print('! Macro AUC gần ngẫu nhiên — bình thường với pilot rất ngắn, nhưng nếu đã chạy '
          'full thì kiểm tra loss, cache guidance/teacher và nhãn yếu trước khi nộp.')
del predictor

INFO:knee_mri.training.checkpoint:Đã nạp checkpoint student_best.pt (epoch 1, best=0.5388)


08:44:13 | INFO    | knee_mri.training.checkpoint | Đã nạp checkpoint student_best.pt (epoch 1, best=0.5388)


INFO:knee_mri.inference.predictor:Predictor sẵn sàng trên cuda từ student_best.pt


08:44:13 | INFO    | knee_mri.inference.predictor | Predictor sẵn sàng trên cuda từ student_best.pt
58 study gold
macro AUC = 0.5372 (trên 12 nhãn)
    Effusion           0.7230
    MCL                0.6440
    ACL                0.6054
    Synovitis          0.5615
    Lateral Meniscus   0.5578
    PF OA              0.5573
    Baker's            0.5543
    Medial OA          0.5271
    Lateral OA         0.4816
    Fracture           0.4542
    Contusion          0.3968
    Medial Meniscus    0.3834
! Macro AUC gần ngẫu nhiên — bình thường với pilot rất ngắn, nhưng nếu đã chạy full thì kiểm tra loss, cache guidance/teacher và nhãn yếu trước khi nộp.


## S3 — Giải thích và sinh submission

In [21]:
# 17. CAM + báo cáo, đối chiếu với mask S1 — ưu tiên study gold (đã chắc chắn có mask)
from knee_mri.pipeline import explain_studies

explain_uids = (split.val + split.train)[:20]
results = explain_studies(cfg, catalog, explain_uids, use_vlm=True, device='cuda')
for item in results[:3]:
    print(item.study_uid[:24], '| IoU CAM/ROI:', item.overlap)

INFO:knee_mri.training.checkpoint:Đã nạp checkpoint student_best.pt (epoch 1, best=0.5388)


08:46:07 | INFO    | knee_mri.training.checkpoint | Đã nạp checkpoint student_best.pt (epoch 1, best=0.5388)


INFO:knee_mri.inference.predictor:Predictor sẵn sàng trên cuda từ student_best.pt


08:46:07 | INFO    | knee_mri.inference.predictor | Predictor sẵn sàng trên cuda từ student_best.pt


INFO:knee_mri.models.report_generator:Đang nạp VLM sinh report google/gemma-3-12b-it...


08:46:07 | INFO    | knee_mri.models.report_generator | Đang nạp VLM sinh report google/gemma-3-12b-it...


... (13 outputs left out)

08:46:10 | INFO    | httpx                        | HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-12b-it "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

INFO:knee_mri.pipeline.explain_step:S3 hoàn tất: 20 study | IoU CAM/ROI trung bình = 0.089


08:53:31 | INFO    | knee_mri.pipeline.explain_step | S3 hoàn tất: 20 study | IoU CAM/ROI trung bình = 0.089
1.2.826.0.1.3680043.8.49 | IoU CAM/ROI: OverlapReport(IoU=0.135, Dice=0.238)
1.2.826.0.1.3680043.8.49 | IoU CAM/ROI: OverlapReport(IoU=0.107, Dice=0.193)
1.2.826.0.1.3680043.8.49 | IoU CAM/ROI: OverlapReport(IoU=0.116, Dice=0.208)


In [22]:
# 18. Sinh submission.csv (CHỈ ảnh + metadata). Chỉ chạy ở MODE='full' — pilot chưa đủ
#     tốt để nộp và dự đoán cả tập test rất tốn thời gian.
if MODE == 'full':
    from knee_mri.inference import predict_submission
    import pandas as pd
    submission_path = predict_submission(cfg, device='cuda')
    print('Submission:', submission_path)
    display(pd.read_csv(submission_path).head())
else:
    print("MODE='pilot' → bỏ qua. Đổi sang 'full', Restart runtime và chạy lại để nộp bài.")

MODE='pilot' → bỏ qua. Đổi sang 'full', Restart runtime và chạy lại để nộp bài.


## Ước lượng cho bản đầy đủ

Chạy sau pilot (khi `timer.seconds` vừa được đo từ đầu — nếu một bước bị bỏ qua vì đã
có cache thì thời gian của nó thấp giả tạo).

In [23]:
# 19. Ngoại suy thời gian từ pilot cho toàn bộ dataset
from knee_mri.pipeline import estimate_full_run

FULL_EPOCHS = 30
hours = estimate_full_run(
    timer.seconds,
    pilot_studies=len(subset),
    full_studies=len(all_studies),
    epoch_seconds=epoch_seconds,
    pilot_train=len(split.train),
    full_train=len(all_studies) - n_gold,
    full_epochs=FULL_EPOCHS,
)
print(f'Pilot: {len(subset)} study | Full: {len(all_studies)} study')
for stage, value in hours.items():
    if stage != 'total':
        print(f'  {stage:<14s} ≈ {value:5.1f} giờ')
print(f'Tổng ≈ {hours["total"]:.1f} giờ GPU A100 (ước lượng thô; chia thành nhiều phiên được vì có resume).')

Pilot: 208 study | Full: 4407 study
  masks          ≈  84.8 giờ
  weak_labels    ≈  14.5 giờ
  teacher_feats  ≈   2.8 giờ
  guidance       ≈   1.9 giờ
  train          ≈  23.8 giờ
Tổng ≈ 127.8 giờ GPU A100 (ước lượng thô; chia thành nhiều phiên được vì có resume).


## Tiếp theo

Mở [`03_results_visualization.ipynb`](03_results_visualization.ipynb) để xem
phân bố dự đoán, CAM overlay và đường ROC trên tập nhãn gold (đặt `RUN` trùng `MODE`).